# 01. Ingesta de datos internacionales — TSI

## Objetivo

Implementar y validar la ingesta de los conjuntos de datos
internacionales del proyecto Traffic Stability Index (TSI)
en Databricks.

Se utilizarán archivos Parquet previamente procesados
y limpiados durante la etapa de ingeniería de datos.

## Fuentes

- Estambul
- São Paulo
- Madrid
- METR-LA
- PEMS-BAY

## Estrategia

1. Verificar la disponibilidad de los archivos.
2. Inspeccionar su estructura.
3. Validar el número de observaciones.
4. Convertir los datos a Delta Lake.
5. Comprobar la integridad de los registros.

Los archivos originales no serán modificados.

In [0]:
ruta = "/Volumes/workspace/tsi/datasets/"

for archivo in dbutils.fs.ls(ruta):
    print(archivo.name, archivo.size, "bytes")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE DATOS DE ESTAMBUL
# ==========================================

RUTA_ISTANBUL = (
    "/Volumes/workspace/tsi/datasets/"
    "istanbul_traffic_index_clean.parquet"
)

# Leer archivo original
df_istanbul = spark.read.parquet(RUTA_ISTANBUL)

# Revisar estructura
print("=== ESTRUCTURA DEL DATASET ===")
df_istanbul.printSchema()

# Mostrar primeros registros
print("\n=== PRIMEROS 5 REGISTROS ===")
df_istanbul.show(5, truncate=False)

# Validar cantidad
total = df_istanbul.count()

print("\n=== VALIDACIÓN DE REGISTROS ===")
print(f"Registros encontrados: {total:,}")
print("Registros esperados: 3,332")

if total == 3332:
    print("VALIDACIÓN CORRECTA")
else:
    print("ADVERTENCIA: revisar cantidad de registros")

## 2. Persistencia en Delta Lake

Se almacena el conjunto limpio de Estambul en una tabla
Delta administrada por Unity Catalog.

La operación conserva las columnas y los registros del
archivo Parquet original, sin aplicar transformaciones
adicionales.

La tabla resultante se utilizará para consultas,
validaciones y posteriores procesos analíticos.

In [0]:
# ==========================================
# TSI - INGESTA DE ESTAMBUL A DELTA LAKE
# Compatible con Databricks Serverless
# ==========================================

TABLA_DESTINO = "workspace.tsi.istanbul_clean"

# Comprobar si la tabla ya existe
if spark.catalog.tableExists(TABLA_DESTINO):
    print(f"La tabla {TABLA_DESTINO} ya existe.")
    print("No se realizó ninguna modificación.")

else:
    (
        df_istanbul.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print("Tabla Delta creada correctamente.")

print(f"Tabla destino: {TABLA_DESTINO}")

In [0]:
# ==========================================
# VALIDACIÓN DE PARQUET VS DELTA
# ==========================================

df_delta = spark.table(
    "workspace.tsi.istanbul_clean"
)

total_parquet = df_istanbul.count()
total_delta = df_delta.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 3332

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql("""
    DESCRIBE DETAIL workspace.tsi.istanbul_clean
""").select(
    "format",
    "numFiles",
    "location"
).show(truncate=False)

In [0]:
# ==========================================
# TSI - VALIDACIÓN DE INTEGRIDAD
# PARQUET VS DELTA
# ==========================================

from pyspark.sql import functions as F

# Comparar estructura
assert df_istanbul.schema == df_delta.schema, (
    "Los esquemas no coinciden."
)

# Comparar contenido, incluyendo duplicados
diferencias_origen = (
    df_istanbul.exceptAll(df_delta).count()
)

diferencias_destino = (
    df_delta.exceptAll(df_istanbul).count()
)

print("=== VALIDACIÓN DE INTEGRIDAD ===")
print("Diferencias Parquet → Delta:", diferencias_origen)
print("Diferencias Delta → Parquet:", diferencias_destino)

assert diferencias_origen == 0
assert diferencias_destino == 0

print("\nVALIDACIÓN COMPLETA")
print("Esquema, cantidad y contenido coinciden.")

In [0]:
# ==========================================
# TSI - INSPECCIÓN DE SÃO PAULO
# ==========================================

RUTA_BASE = "/Volumes/workspace/tsi/datasets"

# Identificar el nombre exacto del archivo
for archivo in dbutils.fs.ls(RUTA_BASE):
    print(f"{archivo.name} | {archivo.size:,} bytes")

## 3. Ingesta de São Paulo

Se inspecciona el conjunto de datos limpio de São Paulo
para comprobar su estructura, cantidad de registros y
compatibilidad con Databricks.

La fuente original se conserva sin modificaciones.

In [0]:
# ==========================================
# TSI - VALIDACIÓN SÃO PAULO
# PARQUET COMPATIBLE CON DATABRICKS
# ==========================================

RUTA_SAO_PAULO = (
    "/Volumes/workspace/tsi/datasets/"
    "sao_paulo_cet_clean_databricks.parquet"
)

df_sao_paulo = spark.read.parquet(RUTA_SAO_PAULO)

print("=== ESQUEMA ===")
df_sao_paulo.printSchema()

print("\n=== TOTAL DE REGISTROS ===")
total = df_sao_paulo.count()

print(f"Registros encontrados: {total:,}")
print("Registros esperados: 5,223,411")

assert total == 5_223_411, (
    "ERROR: La cantidad de registros no coincide."
)

print("\nVALIDACIÓN CORRECTA")

In [0]:
# ==========================================
# TSI - CREACIÓN DE TABLA DELTA SÃO PAULO
# ==========================================

TABLA_DESTINO = "workspace.tsi.sao_paulo_clean"

if spark.catalog.tableExists(TABLA_DESTINO):
    print(f"La tabla {TABLA_DESTINO} ya existe.")
    print("No se realizó ninguna modificación.")

else:
    (
        df_sao_paulo.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print("Tabla Delta creada correctamente.")

print(f"Tabla destino: {TABLA_DESTINO}")

In [0]:
# ==========================================
# TSI - VALIDACIÓN DELTA SÃO PAULO
# ==========================================

df_delta_sao_paulo = spark.table(
    "workspace.tsi.sao_paulo_clean"
)

total_parquet = df_sao_paulo.count()
total_delta = df_delta_sao_paulo.count()

print("=== COMPARACIÓN ===")
print(f"Parquet: {total_parquet:,}")
print(f"Delta:   {total_delta:,}")

assert total_parquet == total_delta == 5_223_411

print("\nVALIDACIÓN CORRECTA")
print("Los conteos coinciden.")

print("\n=== FORMATO DE ALMACENAMIENTO ===")

spark.sql(
    "DESCRIBE DETAIL workspace.tsi.sao_paulo_clean"
).select(
    "format",
    "numFiles",
    "location"
).show(truncate=False)